In [ ]:
from datetime import datetime
import gzip
from io import StringIO
import json
import os
from typing import Literal

import pandas as pd
import requests
from tqdm import tqdm

from kibad_crawler import download_file, s2_api
from kibad_crawler.config import MAX_NUM_BULK_PAPERS, RAW_DATA_DIR, S2_API_KEY

In [ ]:
s2_client = s2_api.S2Client()

# Übersicht relevante Journals und Archive

## Publication Venues

Download [Übersicht relevante Journals und Archive](https://docs.google.com/spreadsheets/d/1ugPuiRNmcVYWcV98TvXsyqVwJYhmk2iZjZ4o9estM2Y/edit?gid=1647072097#gid=1647072097) as CSV and save it to `RAW_DATA_DIR`

In [ ]:
df_venues = s2_client.get_publication_venues()

publication_venue_ids = s2_api.get_publication_venue_id_from_issn_codes(
    file_path=RAW_DATA_DIR
    / "Übersicht relevante Journals und Archive - Ergebnis systematische Literatursuche.csv",
    df_venues=df_venues,
)

# Semantic Scholar

## Papers

In [ ]:
release_path = s2_client._download_all_papers(
    publication_venue_ids=publication_venue_ids, n_files=2, persistant=True
)

## Bulk paper download

In [ ]:
paper_files = [x for x in (release_path).glob("*.csv")]
dfs = [pd.read_csv(f) for f in paper_files]
df_all_papers = pd.concat(dfs)
print(f"{len(df_all_papers):,} papers in total")

In [ ]:
df_all_papers.head()

In [ ]:
paper_ids = [x.split("/")[-1] for x in df_all_papers["url"]]
print(f"{len(paper_ids):,} paper ids")

In [ ]:
json_path = release_path / "json"
json_path.mkdir(parents=True, exist_ok=True)

for i, group in tqdm(
    enumerate(s2_api.chunker(paper_ids, MAX_NUM_BULK_PAPERS)),
    total=int(len(paper_ids) / MAX_NUM_BULK_PAPERS),
):
    response_batch = s2_client.download_bulk_paper(group)

    json_file = json_path / f"{s2_client.latest_release}-{i}.json"
    with open(str(json_file), "w", encoding="utf-8") as f:
        json.dump(response_batch.json(), f)

In [ ]:
def download_paper_batch(
    url: str,
    publication_venue_ids: list,
    persistant: bool = False,
    start_date: str = "2023-01-01",
) -> None:
    file_name = url.split("?")[0].split("/")[-1]
    if (release_path / "papers" / str(file_name + ".csv")).exists():
        return

    file_path = download_file(url=url, path=release_path / "papers")
    file_content = ""
    with gzip.open(
        file_path,
        "r",
    ) as f:
        for line in tqdm(f.readlines(), leave=True):
            if any(x in line.decode() for x in publication_venue_ids):
                file_content += line.decode()

    df_papers = pd.read_json(StringIO(file_content), lines=True)

    date_filter = datetime.strptime(start_date, "%Y-%m-%d")
    df_papers["publicationdate"] = pd.to_datetime(df_papers["publicationdate"])
    df_papers = df_papers[df_papers["publicationdate"] >= date_filter]

    df_papers.to_csv(release_path / "papers" / str(file_name + ".csv"), index=False)

    if not persistant:
        os.remove(file_path)


def download_all_papers(
    file_list: list, publication_venue_ids: list, persistant: bool = False
) -> None:
    for url in tqdm(file_list, total=len(file_list)):
        download_paper_batch(
            url=url, publication_venue_ids=publication_venue_ids, persistant=persistant
        )

## Bulk paper download

In [ ]:
paper_files = [x for x in (release_path / "papers").glob("*.csv")]
dfs = [pd.read_csv(f) for f in paper_files]
df_all_papers = pd.concat(dfs)
print(f"{len(df_all_papers):,} papers in total")

In [ ]:
MIN_NUMBER_OF_CITATIONS: int = 2

df_all_papers_citations = df_all_papers[df_all_papers["citationcount"] > MIN_NUMBER_OF_CITATIONS]

In [ ]:
paper_ids = [x.split("/")[-1] for x in df_all_papers_citations["url"]]
print(f"{len(paper_ids):,} paper ids")

In [ ]:
json_path = release_path / "json"
json_path.mkdir(parents=True, exist_ok=True)

for i, group in tqdm(
    enumerate(s2_api.chunker(paper_ids[:2_000], MAX_NUM_BULK_PAPERS)),
    total=int(2_000 / MAX_NUM_BULK_PAPERS),
):
    response_batch = s2_api.download_bulk_paper(group)

    json_file = json_path / f"{release_path.parent.name}-{i}.json"
    with open(str(json_file), "w", encoding="utf-8") as f:
        json.dump(response_batch.json(), f)

## Read json files

In [ ]:
paper_files = [x for x in (release_path / "json").glob("*.json")]
dfs = [pd.read_json(f) for f in paper_files]
df_all_papers = pd.concat(dfs)
print(f"{len(df_all_papers):,} json in total")

# Incremental Updates (WiP)

In [ ]:
response_releases = requests.get("https://api.semanticscholar.org/datasets/v1/release").json()
latest_release = response_releases[-1]
print(f"{response_releases[-1]} latest release")

In [ ]:
response_releases[-3:]

In [ ]:
release_path = RAW_DATA_DIR / "semantic_scholar" / "release" / f"{latest_release}"
release_path.mkdir(exist_ok=True, parents=True)

In [ ]:
URL_INCREMENTAL = "https://api.semanticscholar.org/datasets/v1/diffs/{start_release_id}/to/{end_release_id}/{dataset_name}"


def download_incremental(
    start_release_id: str,
    end_release_id: str,
    dataset_name: Literal[
        "abstracts",
        "authors",
        "citations",
        "embeddings-specter_v1",
        "embeddings-specter_v2",
        "paper-ids",
        "papers",
        "publication-venues",
        "s2orc",
        "s2orc_v2",
        "tldrs",
    ],
) -> requests.Response:
    headers = {"x-api-key": S2_API_KEY}
    response = requests.get(
        URL_INCREMENTAL.format(
            start_release_id=start_release_id,
            end_release_id=end_release_id,
            dataset_name=dataset_name,
        ),
        headers=headers,
        timeout=60,
    )
    return response


response = download_incremental(
    start_release_id="2026-01-27", end_release_id="2026-02-10", dataset_name="papers"
)

In [ ]:
# ['dataset', 'start_release', 'end_release', 'diffs']
len(response.json()["diffs"][0]["update_files"])

In [ ]:
response.json()["diffs"][0]["update_files"][0]